In [93]:
# Carbon Calculator

In [3]:
import pandas as pd
import numpy as np

In [1]:
# Confirm the cement type
cement_type = input("Enter the cement type: ")

# Functional unit
cement_unit_mass = 1 # tonne

Enter the cement type:  opc


In [101]:
# A1 emission: CO2 from raw material extraction
# A2 emission: CO2 from transport
# A3 emission: CO2 from manufacturing

# A1 stage contains 1) Quarrying raw materials, 2) Crushing, 3) Preparing raw meal
# A2 stage transport
# A3 stage contains 1) fuel combustion, 2) chemical reaction


In [63]:
# A1 - Raw Material Extraction

# ---Constants---
# EF for emission factors
EF_diesel    = 2.68   # kgCO₂/L
EF_ANFO      = 0.26   # kgCO₂/kg
EF_electricity = 0.233  # kgCO₂/kWh

# A1.1 Quarrying 

print("Do you have a reference cement type? (yes/no): ")
type_input = input().strip().lower()

cement_components = ["tricalcium silicate 3CaO.SiO2", 
                      "dicalcium silicate 2CaO.SiO2", 
                      "tricalcium aluminate 3CaO.Al2O3", 
                      "tetracalcium aluminoferrite 4CaO.Al2O3.Fe2O3", 
                      "Gypsum"
                     ]

if composition_input == "yes":
    clinker_in_cement = float(input("Enter the clinker in cement (%): "))

elif composition_input == "no":

    # --- first type of clinker's components weights ---
    print("first type is Oridinary Portland Cement (OPC)")
    type_1 = [50,25,10,10,5]
    cement_components = pd.DataFrame({"type 1 (%)":type_1}, index=cement_components)
    clinker_in_cement = 95

# --- CaO weights in each component ---
CaO_in_component = [(56*3/228),(56*2/172),(56*3/270),(56*4/485.7),0]
cement_components.insert(1,"Cao weight",CaO_in_component)
cement_components["Cao weight"]=cement_components["Cao weight"].round(2)
print (cement_components)

# --- calculate CaO weight in clinker ---
CaO_in_clinker = (cement_components["type 1 (%)"] * cement_components["Cao weight"]).sum()
print("CaO weights in clinker {:.2%}.". format(CaO_in_clinker/100))

CaCO3_CaO_ratio = 1.785  # (stoichiometric ratio, as 100/56)
CaCO3_required = cement_unit_mass * clinker_in_cement/100 * CaO_in_clinker/100 * CaCO3_CaO_ratio # unit in tonne
mineral_purity = float(input("Enter the mineral purity (%): "))
limestone_required = float(CaCO3_required / (mineral_purity/100))
print("Limestone required to produce 1 tonne cement: {:.2f} tonne.". format(limestone_required))

# Diesel and blasting
print("\n--- A1.1 Quarrying ---")
raw_materials = ["Limestone", "Clay", "Sand", "Iron ore"]
CO2_extraction = 0

print("Do you have a reference raw materials composition? (yes/no): ")
composition_input = input().strip().lower()

if composition_input == "no":
    OPC_type_1 = [1210, 200, 50, 30]
    diesel_default = [2.0, 1.5, 1.0, 3.0]   # L/tonne
    anfo_default   = [0.18, 0.10, 0.06, 0.22]  # kg/tonne

    mass_df = pd.DataFrame({
        "OPC_type_1 (kg)":   OPC_type_1,
        "Diesel (L/tonne)":  diesel_default,
        "ANFO (kg/tonne)":   anfo_default
    }, index=raw_materials)
    
    print("Using default OPC Type 1 composition:")
    print(mass_df) 
    
    for material in raw_materials:
        mass   = mass_df.loc[material, "OPC_type_1 (kg)"]
        diesel = mass_df.loc[material, "Diesel (L/tonne)"]
        anfo   = mass_df.loc[material, "ANFO (kg/tonne)"]
        
        CO2_diesel   = diesel * mass/1000 * EF_diesel
        CO2_blasting = anfo * mass/1000 * EF_ANFO
        CO2_material = CO2_diesel + CO2_blasting
        CO2_extraction += CO2_material

        print(f"{material} CO₂ subtotal:  {CO2_material:.2f} kgCO₂")

elif composition_input == "yes":
    
    print("""
    Reference table (main raw materials extracted):
    ————————————————————————————————————————————————
    Raw Material    Typical proportion per tonne OPC
    Limestone       1200–1500 kg
    Clay            200–400 kg
    Sand            50–100 kg
    Iron ore        30–50 kg
    ————————————————————————————————————————————————
    """)
    
    print("""
    Reference Table (typical values from literature):
    ─────────────────────────────────────────────────────────
    Raw Material    Diesel (L/tonne)    ANFO (kg/tonne)
    Limestone       1.5 – 3.0           0.10 – 0.25
    Clay            1.0 – 2.0           0.05 – 0.15
    Sand            0.5 – 1.5           0.03 – 0.10
    Iron ore        2.0 – 4.0           0.15 – 0.30
    ─────────────────────────────────────────────────────────
    """)

    for material in raw_materials:
        print(f"{material}:")
        mass   = float(input(f"  Enter mass of {material} per tonne OPC (kg): "))
        diesel = float(input(f"  Enter diesel consumed (L/tonne): "))
        anfo   = float(input(f"  Enter ANFO used (kg/tonne): "))
    
        CO2_diesel = diesel * mass/1000 * EF_diesel
        CO2_blasting = anfo * mass/1000 * EF_ANFO
        CO2_material = CO2_diesel + CO2_blasting
        CO2_extraction += CO2_material
        
        print(f"  CO₂ subtotal: {CO2_material:.2f} kgCO₂")
        
# A1.2 Crushing
print("\n--- A1.2 Crushing & Screening ---")
print("""
Reference Table (typical values from literature):
─────────────────────────────────────────────────
Equipment           Power (kW)      hrs/tonne
Primary crusher     200 – 500       0.5 – 1.5
Secondary crusher   100 – 300       0.5 – 1.0
Screening unit      50 – 150        0.3 – 0.8
─────────────────────────────────────────────────
""")

crushing_equipment = ["Primary crusher", "Secondary crusher", "Screening unit"]
total_crushing_energy = 0

for eq in crushing_equipment:
    print(f"{eq}:")
    power = float(input("  Enter power (kW): "))
    hours = float(input("  Enter operating hours per tonne (hrs/tonne): "))
    energy = power * hours
    total_crushing_energy += energy
    print(f"  Energy:  {energy:.2f} kWh/tonne")

CO2_crushing = total_crushing_energy * EF_electricity

print(f"  Total crushing energy:  {total_crushing_energy:.2f} kWh")
print(f"  CO₂ crushing:           {CO2_crushing:.2f} kgCO₂")


CO2_A1 = CO2_extraction + CO2_crushing

print("\n A1 Raw Material Extraction Summary:")
print(f"  Extraction:           {CO2_extraction:.2f} kgCO₂")
print(f"  Crushing & screening: {CO2_crushing:.2f} kgCO₂")
print(f"  TOTAL A1:             {CO2_A1:.2f} kgCO₂/tonne OPC")

Do you have a reference cement type? (yes/no): 


 no


first type is Oridinary Portland Cement (OPC)
                                              type 1 (%)  Cao weight
tricalcium silicate 3CaO.SiO2                         50        0.74
dicalcium silicate 2CaO.SiO2                          25        0.65
tricalcium aluminate 3CaO.Al2O3                       10        0.62
tetracalcium aluminoferrite 4CaO.Al2O3.Fe2O3          10        0.46
Gypsum                                                 5        0.00
CaO weights in clinker 64.05%.


Enter the mineral purity (%):  90


Limestone required to produce 1 tonne cement: 1.21 tonne.

--- A1.1 Quarrying ---
Do you have a reference raw materials composition? (yes/no): 


...

  Energy:  300.00 kWh/tonne
Secondary crusher:


  Enter power (kW):  100
  Enter operating hours per tonne (hrs/tonne):  1


  Energy:  100.00 kWh/tonne
Screening unit:


  Enter power (kW):  50
  Enter operating hours per tonne (hrs/tonne):  0.8


  Energy:  40.00 kWh/tonne
  Total crushing energy:  440.00 kWh
  CO₂ crushing:           102.52 kgCO₂

 A1 Raw Material Extraction Summary:
  Extraction:           7.73 kgCO₂
  Crushing & screening: 102.52 kgCO₂
  TOTAL A1:             110.25 kgCO₂/tonne OPC


In [33]:
# Calculate energy consumed to run crusher 
# Step 1 - jaw crusher - primary crusher used to break large limestone blocks into smaller pieces
# reference jaw crusher is 600t/hr, 220kW 
# https://www.mining-rockcrusher.com/sale-15458633-92000kgs-heavy-duty-jaw-crusher-machine-granite-limestone-crushing.html
crusher_capacity_1 = float(input("Enter the crusher 1 capacity (t/hr): "))
#time_crushing_1_tonne = float(1/crusher_capacity)  # (unit in hrs)
crusher_power_1 = float(input("Enter the crusher 1 power (KW): "))

# Step 2 - Impact crusher / hammer crusher - secondary crushing produces sand-like particles
# reference jaw crusher is 600t/hr, 220kW
crusher_capacity_2 = float(input("Enter the crusher 2 capacity (t/hr): "))
crusher_power_2 = float(input("Enter the crusher 2 power (KW): "))

def energy_crushing (crusher_capacity,crusher_power):
    return (1/crusher_capacity) * crusher_power  # (unit in KWh)

Jaw_crusher_energy = energy_crushing (crusher_capacity_1,crusher_power_1)
hammer_crusher_energy = energy_crushing (crusher_capacity_2,crusher_power_2)
energy_crushing_1_tonne = Jaw_crusher_energy + hammer_crusher_energy

total_energy_crushing = limestone_required * energy_crushing_1_tonne
print("Total energy to crush limestone per 1 tonne cement produced: {:.3f} KWh.".format(total_energy_crushing))

KeyboardInterrupt: Interrupted by user

In [113]:
# define a dataframe for fuel mix in grid network and emission intensity
fuel_mix = pd.DataFrame(columns=["Coal","Natural Gas","Nuclear","Renewables","Other"])

# Input one row representing fuel mix at that location
# UK grid energy sources with CO2 emission from fuel mix 
# UK Reference: https://www.gov.uk/government/publications/fuel-mix-disclosure-data-table/fuel-mix-disclosure-data-table#fn:1
UK_fuel_mix = [5.9, 33.3, 16.2, 42.1, 2.5] # (total 100%)
UK_CO2_intensity_from_fuel_mix = [0.919,0.382,0,0,0.948] # (in kg/KWh)
fuel_mix.loc["UK (%)"] = UK_fuel_mix
fuel_mix.loc["CO2 (kg/KWh)"] = UK_CO2_intensity_from_fuel_mix
print (fuel_mix)

# Average CO2 emission from UK grid mix = Grid mix × CO2 emission from energy mix
UK_average_CO2_fuel_mix = ((fuel_mix.loc["UK (%)"]/100) * fuel_mix.loc["CO2 (kg/KWh)"]).sum()  # CO2 emission per 1KWh energy
print("Average CO2 emission from UK grid: {:.5f} kg CO2eq./KWh.".format(UK_average_CO2_fuel_mix))

               Coal  Natural Gas  Nuclear  Renewables  Other
UK (%)        5.900       33.300     16.2        42.1  2.500
CO2 (kg/KWh)  0.919        0.382      0.0         0.0  0.948
Average CO2 emission from UK grid: 0.20513 kg CO2eq./KWh.


In [115]:
# Calculate CO2 emission for crushing limestone per 1 tonne cement produced

# Calculate CO2 emission from crushing limestone
CO2_emission_crushing_limestone = UK_average_CO2_fuel_mix * total_energy_crushing
print("CO2 emission from crushing limestone per 1 tonne cement produced: {:.2f} kg CO2eq.".format(CO2_emission_crushing_limestone))

CO2 emission from crushing limestone per 1 tonne cement produced: 0.18 kg CO2eq.


In [117]:
# A2 - Transportation

# --- Constants ---
EF_truck         = 0.062   # kgCO₂/tonne·km
EF_electric_rail = 0.022   # kgCO₂/tonne·km
EF_diesel_rail   = 0.041   # kgCO₂/tonne·km

print("\n--- A2 Transportation ---")

limestone_mass = mass_df.loc["Limestone", "OPC_type_1 (kg)"] / 1000
clay_mass      = mass_df.loc["Clay",      "OPC_type_1 (kg)"] / 1000
sand_mass      = mass_df.loc["Sand",      "OPC_type_1 (kg)"] / 1000
ironore_mass   = mass_df.loc["Iron ore",  "OPC_type_1 (kg)"] / 1000

total_raw_mass = limestone_mass + clay_mass + sand_mass + ironore_mass
print(f"\n  Total raw material mass to transport: {total_raw_mass:.4f} tonne")

# Formula: CO₂ = mass (tonne) × distance (km) × EF (kgCO₂/tonne·km)

# Path 1 - Quarry to Cement Plant
print("Path 1: Quarry → Cement Plant")
print("Available transport modes: truck, conveyor, electric_rail, diesel_rail")
mode_path1 = input("Enter transport mode: ").strip().lower()

if mode_path1 == "conveyor":
    power_conveyor = float(input("Enter conveyor power (kW): "))
    hours_conveyor = float(input("Enter conveyor operating hours (hrs): "))
    CO2_path1 = power_conveyor * hours_conveyor * EF_electricity
    print(f"Energy consumed: {power_conveyor * hours_conveyor:.2f} kWh")
else:
    distance_path1 = float(input("Enter distance (km): "))
    if mode_path1 == "truck":
        ef = EF_truck
    elif mode_path1 == "electric_rail":
        ef = EF_electric_rail
    elif mode_path1 == "diesel_rail":
        ef = EF_diesel_rail

CO2_leg1 = total_raw_mass * distance_path1 * ef

print(f"CO₂ path 1: {CO2_leg1:.2f} kgCO₂")

# Path 2 - Fuel Supplier to Cement Plant


# Path 3 - Gypsum Supplier to Cement Plant
print("Path 3: Gypsum Supplier → Cement Plant (truck)")
distance_leg3     = float(input("Enter distance (km): "))
gypsum_proportion = 0.05  # assumption: 5% of clinker
gypsum_mass       = cement_unit_mass * (clinker_in_cement/100) * gypsum_proportion  # tonne
CO2_leg3          = distance_leg3 * gypsum_mass * EF_truck

print(f"  Gypsum mass:        {gypsum_mass:.4f} tonne")
print(f"  CO₂ Leg 3:          {CO2_leg3:.2f} kgCO₂")

CO2 emission from transporting limestone per 1 tonne cement produced: 5.97 kg CO2eq.


In [119]:
# Explosives emission
# Amount of explosive used per 1 tonne limestone extracted × Limestone required for 1 tonne cement × Emission factor of explosive
explosives_amount = float(input("Enter amount of explosives used per 1 tonne limestone extracted (kg): "))
emission_factor_explosives = 0.1
CO2_emission_explosives = float(explosives_amount * limestone_required * emission_factor_explosives)
print("CO2 emission from explosives per 1 tonne cement produced: {:.2f} kg CO2eq.".format(CO2_emission_explosives))

Enter amount of explosives used per 1 tonne limestone extracted (kg):  100


CO2 emission from explosives per 1 tonne cement produced: 11.94 kg CO2eq.


In [121]:
#sum A1 emission
A1_emission = a1_emission(CO2_emission_transport_1,CO2_emission_explosives,CO2_emission_crushing_limestone)
print("A1 emission, CO2 from raw material supply per 1 tonne cement produced: {:.2f} kg CO2eq.".format(A1_emission))

A1 emission, CO2 from raw material supply per 1 tonne cement produced: 18.09 kg CO2eq.


In [123]:

def a3_emission (roller_mill_blender, CO2_process, fuel_combustion, cooling_fan, conveyor):
    return roller_mill_blender + CO2_process + fuel_combustion + cooling_fan + conveyor

In [129]:
# A2 transport

# Limestone required / Truck loading capacity × Fuel consumption per truck per km × Transporting distance × CO2 emission per 1L diesel

# Truck loading capacity 
truck_loading = float(input("truck loading capacity (tonne): "))
fuel_consumption_per_truck_100km = 30 # in l/100km
# Transportation distance from quarrying site to processing unit
distance_2 = 300 # in km, from storage unit to processing plant
CO2_emission_diesel = 3 # in kg/l
# Calculate the CO2 emission from transportation
CO2_emission_transport_2 = float (limestone_required / truck_loading * (fuel_consumption_per_truck_100km / 100) * distance_2 * CO2_emission_diesel)
A2_emission = CO2_emission_transport_2
print("A2 CO2 emission per 1 tonne cement produced: {:.2f} kg CO2eq.". format(A2_emission))

truck loading capacity (tonne):  25


A2 CO2 emission per 1 tonne cement produced: 12.89 kg CO2eq.


In [131]:
# Roller mill blender operation
# Time per blending 1 ton raw materials × raw materials required for 1 ton cement × Energy consumed to run roller mill blender × CO2 emission per 1KWh energy
CO2_roller_mill_blender = 3 # in kg CO2eq

# CO2/CaCO3 Stoichiometric Ratio
CO2_CaCO3_ratio = 0.44
CO2_chemical = CO2_CaCO3_ratio * CaCO3_required * 1000
print("CO2 emission in chemical reaction to produce 1 tonne cement: {:.2f} kg.". format(CO2_chemical))

CO2 emission in chemical reaction to produce 1 tonne cement: 472.67 kg.


In [151]:
# Fuel combustion
# Total heat ≈ sensible heat to raise solids + calcination heat + losses.
# heat requirement in modern dry kilns is typically 3.0–3.4 GJ per tonne of clinker
total_heat = 3000 #unit in MJ
# fuel volume = total heat / fuel lower heating value
natural_gas_LHV = 36.6 # unit in MJ/m3
ng_volume = total_heat / natural_gas_LHV

biodiesel_LHV = 34.8 # unit in MJ/l
biodiesel_volume = total_heat / biodiesel_LHV

# ghg conversion factor is used to calculate emissions of different fuels
ng_ghg = 2.06 #unit in kgCO2/m3
biodiesel_ghg = 2.57 #unit in kgCO2/l

CO2_ng = ng_volume * ng_ghg # in kgCO2eq
CO2_biodiesel = biodiesel_volume * biodiesel_ghg
print ("{:.2f} kg". format (CO2_ng))
print ("{:.2f} kg". format (CO2_biodiesel))

#cooling_fan
CO2_cooling_fan = 0 # in kg CO2eq

#conveyor
CO2_conveyor = 0 # in kg CO2eq

A3_emission = a3_emission (CO2_roller_mill_blender, CO2_chemical, CO2_ng, CO2_cooling_fan, CO2_conveyor)
print("A3 CO2 emission per 1 tonne cement produced: {:.2f} kg CO2eq.". format(A3_emission))

168.85 kg
221.55 kg
A3 CO2 emission per 1 tonne cement produced: 644.53 kg CO2eq.


In [153]:
A1_A3_emission = A1_emission + A2_emission + A3_emission

Emission_summary = pd.DataFrame({"Emission (kg CO2)":[A1_emission,A2_emission,A3_emission,A1_A3_emission]}, index=["A1", "A2", "A3", "S1"])
Emission_summary["Emission (kg CO2)"] = Emission_summary["Emission (kg CO2)"].round(2)

Emission_summary["share %"] = Emission_summary["Emission (kg CO2)"]/A1_A3_emission*100
Emission_summary["share %"] = Emission_summary["share %"].round(1)
print(Emission_summary)

    Emission (kg CO2)  share %
A1              18.09      2.7
A2              12.89      1.9
A3             644.53     95.4
S1             675.50    100.0


In [149]:
def carbon_calculator(A1_emission, A2_emission, A3_emission):
    return A1_emission + A2_emission + A3_emission
Total_carbon = carbon_calculator (A1_emission, A2_emission, A3_emission)
print(f"{cement_type} produces {Total_carbon:.2f} kg CO2eq per tonne cement.")

opc produces 675.50 kg CO2eq per tonne cement.
